# 04 · Dataset Builder · XAUUSD

Vérifier Gold, joindre les horizons déjà clôturés, définir les fenêtres et ajuster les transformations exclusivement sur Train.

Au démarrage, `SNAPSHOT` présente les résultats **FULL du 1er janvier 2004 au 23 novembre 2026** disponibles dans `data/pipeline_runs/full_notebooks/`. `TEST` utilise une fixture, `PILOT` une période bornée et `FULL` les dates du 1er janvier 2004 au 23 novembre 2026 inclus. `RESUME` utilise le périmètre indiqué par `XAUUSD_RESUME_MODE`. Conserver le même `XAUUSD_RUN_LABEL` dans les quatre notebooks. Les sorties sont en Parquet ZSTD ou JSON, sous `data/pipeline_runs/`.

Spark fonctionne sous Linux ou WSL2 avec Java 17. Les tâches exécutent en parallèle les partitions de calcul. Les échanges entre tâches utilisent le disque temporaire hors OneDrive. Les partitions physiques Parquet permettent aux lectures ultérieures de réduire le périmètre lu.

**Suivi du FULL** · Les graphiques se rafraîchissent à chaque exécution des cellules de visualisation. Ils utilisent les checkpoints compatibles déjà terminés. Les mois gris dans les cartes de couverture n’ont aucune mesure publiée à afficher. Pour lancer ou reprendre les traitements, utiliser `XAUUSD_MODE=FULL` ou `XAUUSD_MODE=RESUME` sous WSL2.

## Configuration et ressources

Le nombre de workers et la mémoire du driver dépendent des ressources disponibles. UTC stabilise les timestamps Spark. AQE adapte les partitions après shuffle. La dernière cellule consulte les métriques disponibles de Spark UI.

In [1]:
STAGE = 'dataset'
from collections import defaultdict, deque
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from hashlib import sha256
from importlib.util import find_spec
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import csv, gzip, json, math, os, platform, re, shutil, subprocess, sys
from IPython.display import Markdown, display

def locate_project():
    candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / 'OneDrive/Desktop/Multi-Agents-Trading',
                  *sorted(Path('/mnt/c/Users').glob('*/OneDrive/Desktop/Multi-Agents-Trading'))]
    return next((p for p in candidates if (p / 'data').is_dir() and (p / 'pipelinebigdata').is_dir()), Path.cwd())

SMA = EMA = [5, 10, 20, 50]
CFG = dict(
    mode=os.getenv('XAUUSD_MODE', 'SNAPSHOT').upper(),
    resume_mode=os.getenv('XAUUSD_RESUME_MODE', 'FULL').upper(),
    root=Path(os.getenv('XAUUSD_PROJECT_DIR', str(locate_project()))).resolve(),
    start=os.getenv('XAUUSD_FROM'), end=os.getenv('XAUUSD_TO'),
    scratch=Path(os.getenv('XAUUSD_SPARK_TMP', str(Path.home() / '.cache/xauusd-spark'))).resolve(),
    steps=set(os.getenv('XAUUSD_STEPS', '02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18,19,20').split(',')),
    source_ids=['historical_old', 'historical_recent', 'touhid', 'carlos', 'jforex'],
    timeframes=['M1','M5','M15','M30','H1','H4','D1','W1','MN1'],
    main='M5', higher=['M15','H1','H4','D1'],
    sma=SMA, ema=EMA, rsi=[7,14], macd=[(5,13,4),(5,26,4)], stoch=[(9,3),(14,3)],
    cci=[14], atr=[7,14], bollinger=[(20,2)], obv=[10,20],
    extra={'price','trend','momentum','volatility','statistics','microstructure','time'},
    extra_window=20, roc=10, adx=14, sar_step=.02, sar_max=.2,
    max_gap_seconds=180, volume_required=False, allowed_quality=['OK'], instrument='XAUUSD',
    dataset_sources=['historical_old','historical_recent','touhid','carlos','jforex','fixture'],
    dataset_start=None, dataset_end=None, features=['sma_5','rsi_7','tick_obv_delta_10'],
    split=None, scaler='standard', lookback=1, benchmark_full=os.getenv('XAUUSD_BENCHMARK_FULL','0')=='1',
    benchmark_levels=os.getenv('XAUUSD_BENCHMARK_LEVELS','day').split(','),
    # Ajouter des périodes HistData uniquement après rapprochement prix/horloge documenté.
    histdata_windows=[('2021-01-05','2021-01-06',5),('2025-08-05','2025-08-06',4)],
    # Ces deux fenêtres HistData servent à la comparaison, leur admission se règle via source_ids.
)
data = Path(os.getenv('XAUUSD_DATA_DIR', str(CFG['root'] / 'data'))).resolve()
started_at = perf_counter()
mode = CFG['mode']
run_mode = CFG['resume_mode'] if mode in {'RESUME','SNAPSHOT'} else mode
assert run_mode in {'TEST','PILOT','FULL'} and mode in {'TEST','PILOT','FULL','RESUME','SNAPSHOT'}
if run_mode == 'FULL':
    start, end = date(2004,1,1), date(2026,11,24)
elif run_mode == 'TEST':
    start, end = date(2025,8,5), date(2025,8,6)
else:
    if not CFG['start'] or not CFG['end']:
        raise ValueError('Le mode PILOT demande XAUUSD_FROM et XAUUSD_TO (borne UTC exclue).')
    start, end = map(date.fromisoformat, (CFG['start'], CFG['end']))
assert start < end
if CFG['instrument']!='XAUUSD':
    raise ValueError('Le catalogue installé couvre XAUUSD.')
run_folder = {'FULL': 'full', 'TEST': 'test', 'PILOT': f'pilot_{start}_{end}'}[run_mode]
label = os.getenv('XAUUSD_RUN_LABEL', 'notebooks' if run_mode == 'FULL' else '')
if label and not re.fullmatch(r'[A-Za-z0-9_-]+', label):
    raise ValueError('XAUUSD_RUN_LABEL doit contenir des lettres, chiffres, tirets ou underscores.')
out = data / 'pipeline_runs' / (f'{run_folder}_{label}' if label else run_folder)
reports = out / 'reports' / 'spark_xauusd'
manifests = out / 'manifests' / 'spark_xauusd'
assert 'onedrive' not in str(CFG['scratch']).lower() and not CFG['scratch'].is_relative_to(data)

def say(text):
    display(Markdown(text))

def active(step):
    return mode != 'SNAPSHOT' and step in CFG['steps']

def probe(command):
    try:
        result = subprocess.run(command, text=True, encoding='utf8', errors='replace', capture_output=True, timeout=7)
        return ((result.stdout or result.stderr).replace('\x00','').strip().splitlines() or [''])[0] if result.returncode == 0 else 'indisponible'
    except (OSError, subprocess.TimeoutExpired):
        return 'indisponible'

psutil = __import__('psutil') if find_spec('psutil') else None
ram = psutil.virtual_memory() if psutil else None
cores = len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count() or 2
avail = ram.available / 2**30 if ram else 4
workers = min(cores, max(2, int(avail // 1.5)), 6)
driver_gib = max(1, min(6, int(avail * .55)))
shuffle_partitions = max(128,workers*32) if run_mode=='FULL' else max(8,workers*4)
cpu_name = os.environ.get('PROCESSOR_IDENTIFIER') or platform.processor() or platform.machine()
if Path('/proc/cpuinfo').is_file():
    cpu_name = next((line.split(':',1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines()
                     if line.startswith('model name')),cpu_name)
spark_options = {'spark.driver.memory': f'{driver_gib}g', 'spark.sql.shuffle.partitions': str(shuffle_partitions),
                 'spark.sql.adaptive.enabled':'true', 'spark.sql.parquet.enableVectorizedReader':'true','spark.sql.files.maxPartitionBytes':str(96*2**20),
                 'spark.local.dir':str(CFG['scratch']), 'spark.sql.session.timeZone':'UTC',
                 'spark.driver.bindAddress':'127.0.0.1','spark.driver.host':'127.0.0.1',
                 'spark.sql.parquet.compression.codec':'zstd',
                 'spark.sql.parquet.outputTimestampType':'TIMESTAMP_MICROS',
                 'spark.sql.execution.arrow.pyspark.enabled':'false'}
free_data = shutil.disk_usage(data if data.exists() else Path.cwd()).free / 2**30
print(dict(stage=STAGE, physical_cores=psutil.cpu_count(logical=False) if psutil else None, mode=mode, period=[str(start),str(end)], CPU=cpu_name,
           cores=cores, ram_total_GiB=round(ram.total/2**30,2) if ram else None,
           ram_available_GiB=round(avail,2), disk_available_GiB=round(free_data,2),
           Python=platform.python_version(), Java=probe(['java','-version']),
           PySpark=bool(find_spec('pyspark')), Docker=probe(['docker','--version']),
           WSL2='microsoft' in platform.release().lower() or probe(['wsl.exe','--list','--verbose'])!='indisponible',
           master=f'local[{workers}]', scratch=str(CFG['scratch'])))
spark = None
if mode != 'SNAPSHOT':
    if not find_spec('pyspark'):
        raise RuntimeError('Installer PySpark et Java 17 dans le noyau Linux/WSL2.')
    if os.name == 'nt':
        raise RuntimeError('Utiliser le noyau Linux/WSL2 pour les écritures Parquet Spark.')
    from pyspark.sql import SparkSession, Window, functions as F, types as T
    from pyspark import StorageLevel
    CFG['scratch'].mkdir(parents=True, exist_ok=True)
    os.environ['TZ']='UTC'
    import time
    if hasattr(time,'tzset'):
        time.tzset()
    os.environ.setdefault('SPARK_LOCAL_IP','127.0.0.1')
    os.environ.setdefault('PYSPARK_PYTHON',sys.executable)
    spark = SparkSession.builder.master(f'local[{workers}]').appName(f'XAUUSD {STAGE}').config(
        map=spark_options).getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    print(f'Spark {spark.version} · UI {spark.sparkContext.uiWebUrl}')

def read_json(path, default=None):
    return json.loads(path.read_text(encoding='utf-8-sig')) if path.is_file() else default

def put_json(path, value):
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_name(f'.{path.name}.{uuid4().hex}.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str, allow_nan=False)+'\n', encoding='utf8')
    temp.replace(path)

def hash_file(path):
    # Sur WSL, un gros fichier déjà sur NTFS est haché par Windows sans traverser DrvFS octet par octet.
    if (path.stat().st_size>=2**30 and re.match(r'^/mnt/[a-z]/',str(path))
        and shutil.which('powershell.exe') and shutil.which('wslpath')):
        import base64
        try:
            win = subprocess.check_output(['wslpath','-w',str(path)],text=True,timeout=10).strip()
            command = "$ErrorActionPreference='Stop'; $s=[System.IO.File]::OpenRead('" + win.replace("'","''") + "'); try { $sha=[System.Security.Cryptography.SHA256]::Create(); [Console]::WriteLine([BitConverter]::ToString($sha.ComputeHash($s)).Replace('-','')) } finally { $s.Dispose() }"
            encoded = base64.b64encode(command.encode('utf-16-le')).decode('ascii')
            raw = subprocess.check_output(['powershell.exe','-NoProfile','-NonInteractive',
                                          '-EncodedCommand',encoded],stderr=subprocess.DEVNULL,timeout=900)
            match=re.search(rb'(?i)[0-9a-f]{64}',raw)
            if match:
                return match.group().decode('ascii').lower()
        except (OSError,subprocess.SubprocessError):
            pass
    with path.open('rb') as stream:
        from hashlib import file_digest
        return file_digest(stream,'sha256').hexdigest()

def digest(value):
    return sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

key_config = {k: sorted(v) if isinstance(v,set) else v for k,v in CFG.items()
              if k not in {'mode','resume_mode','root','scratch','steps','benchmark_full','benchmark_levels','start','end'}}
run_key = digest([key_config,str(start),str(end),'xauusd-spark-notebooks-v1'])

def manifest(name):
    return read_json(manifests / (name+'.json'), {})

def completed(name):
    state = manifest(name)
    return state.get('run_key') == run_key and state.get('status') == 'complete'

def status(name):
    record = manifest(name)
    say(f"**{name}** · {record.get('status','aucun résultat sauvegardé')}.")
    if record:
        print({k:v for k,v in record.items() if k not in {'run_key','files','schema'}})

def require(name, path):
    if not completed(name) or not path.is_dir() or read_json(path/'_pipeline.json',{}).get('run_key')!=run_key:
        raise RuntimeError(f'Checkpoint {name} absent ou incompatible : {path}')
    return spark.read.parquet(str(path))

def publish(df, path, partition=()):
    if path.exists():
        if read_json(path/'_pipeline.json',{}).get('run_key')==run_key and (path/'_SUCCESS').exists():
            return
        raise FileExistsError(f'Résultat existant sans manifeste compatible : {path}')
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.{uuid4().hex}.pending')
    writer = df.write.mode('errorifexists').option('compression','zstd')
    if partition:
        writer = writer.partitionBy(*partition)
    writer.parquet(str(tmp))
    if not (tmp / '_SUCCESS').exists():
        raise RuntimeError(f'Écriture Spark incomplète : {tmp}')
    put_json(tmp/'_pipeline.json',{'run_key':run_key})
    tmp.rename(path)

def finish(name, **stats):
    put_json(manifests / (name+'.json'), {'run_key':run_key,'status':'complete','elapsed_run_seconds':round(perf_counter()-started_at,3),**stats})
    status(name)

NORM = out/'silver/normalized_sources'
QUAR = out/'quarantine/spark_xauusd'
TICKS = out/'silver/ticks'
BARS = out/'silver/bars'
GOLD = out/'gold/features'
BUILT = out/'datasets/spark_xauusd'
say('La configuration et les chemins s’appliquent à toutes les cellules suivantes.')

def source_signature(items):
    inventory = []
    for item in items:
        relative = item['source_file_id']
        path = data / relative
        st = path.stat() if path.is_file() else None
        inventory.append([relative, item.get('sha256'), st.st_size if st else None, st.st_mtime_ns if st else None])
    proof = data / 'reports/preuves_decimales_2021-01-05.json'
    return digest([inventory, hash_file(proof) if proof.exists() else None])

def load_bronze():
    global run_key
    contract = read_json(reports/'catalogue_bronze.json', {})
    if not contract or contract.get('version') != 1 or contract.get('base_key') != run_key:
        if mode == 'SNAPSHOT':
            return {}
        raise RuntimeError('Catalogue Bronze absent ou incompatible. Exécuter 01_Bronze avec le même périmètre.')
    if mode != 'SNAPSHOT' and source_signature(contract['sources']) != contract['sources_key']:
        raise RuntimeError('Inventaire source modifié depuis Bronze.')
    run_key = digest([run_key, contract['sources_key']])
    if mode != 'SNAPSHOT' and not completed('bronze'):
        raise RuntimeError('Audit Bronze incomplet dans ce périmètre.')
    print('Catalogue Bronze compatible', len(contract['sources']), 'fichiers')
    return contract

def size_of(path):
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file()) if path.exists() else 0

def stages():
    from urllib.parse import quote
    from urllib.request import urlopen
    try:
        app = quote(spark.sparkContext.applicationId)
        with urlopen(f'{spark.sparkContext.uiWebUrl}/api/v1/applications/{app}/stages',timeout=3) as stream:
            return {row['stageId']:row for row in json.load(stream)}
    except (OSError, ValueError):
        return None

def stage_metrics():
    observed = stages() if spark else None
    if observed is None:
        print('Spark UI : métriques de tâches indisponibles.')
        return {}
    fields = ('numTasks','numFailedTasks','executorCpuTime','jvmGcTime',
              'shuffleReadBytes','shuffleWriteBytes','diskBytesSpilled','memoryBytesSpilled')
    return {'stages':len(observed), **{
        key:sum(value[key] for value in observed.values() if key in value)
        if any(key in value for value in observed.values()) else None
        for key in fields}}

def stage_report():
    groups = {
        'bronze': ('bronze',),
        'silver': ('normalized','silver','validation','bars'),
        'gold': ('gold','causality','catalogue_features'),
        'dataset': ('dataset',),
    }
    stats = {name:manifest(name) for name in groups[STAGE]}
    valid = {name:rec for name,rec in stats.items() if rec.get('run_key')==run_key}
    metrics = stage_metrics()
    if spark:
        put_json(reports/f'observabilite_{STAGE}.json', {
            'run_key':run_key, 'ui':spark.sparkContext.uiWebUrl,
            'seconds':round(perf_counter()-started_at,3), 'metrics':metrics})
    print('Métriques Spark UI',metrics)
    for name, record in stats.items():
        print(name, record if name in valid else 'non exécuté ou incompatible')

{'stage': 'dataset', 'physical_cores': 4, 'mode': 'SNAPSHOT', 'period': ['2004-01-01', '2026-11-24'], 'CPU': 'Intel(R) Core(TM) i5-8250U CPU @ 1.60GHz', 'cores': 8, 'ram_total_GiB': 7.68, 'ram_available_GiB': 5.01, 'disk_available_GiB': 88.69, 'Python': '3.12.3', 'Java': 'openjdk version "17.0.20.1" 2026-08-18', 'PySpark': True, 'Docker': 'indisponible', 'WSL2': True, 'master': 'local[3]', 'scratch': '/home/omar/.cache/xauusd-spark'}


La configuration et les chemins s’appliquent à toutes les cellules suivantes.

## Contrat et données d’entrée

Le catalogue de Bronze fige le périmètre, les chemins relatifs, les conventions et l’identité des fichiers. Les checkpoints sont acceptés uniquement pour la même exécution.

In [2]:
bronze = load_bronze()
D = data/'recovered/autres_fournisseurs'
legacy = {'historical_old':'XAUUSD_Ticks_2004.01.01_2021.10.15.csv',
          'historical_recent':'XAUUSD_Ticks_2021.01.04_2026.11.20.csv'}
sources = bronze.get('sources',[])
legacy_refs = bronze.get('legacy_refs',[])
original_audit = {'comparaison': bronze.get('overlap',{})}
annual = bronze.get('annual',[])
print('Sources du catalogue',len(sources))

Catalogue Bronze compatible 432 fichiers
Sources du catalogue 432


## Fenêtre et lecture des graphiques

Les figures s’appuient sur les manifestes et les métadonnées Parquet. Les comparaisons excluent les checkpoints d’un autre périmètre. La période affichée dans les titres est le périmètre demandé, avec la couverture mesurée indiquée par les données et les cartes.

In [3]:
from collections import Counter
from matplotlib import dates as mdates
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator, StrMethodFormatter

period_label = f'{run_mode} · {start:%d/%m/%Y} au {end-timedelta(days=1):%d/%m/%Y}'
mib = 2**20

def completed_parts(prefix):
    result=[]
    for path in sorted(manifests.glob(prefix+'*.json')):
        state=read_json(path,{})
        if state.get('status')=='complete' and state.get('run_key')==run_key:
            result.append((path.stem.removeprefix(prefix),state))
    return result

def timeline(ax):
    locator=mdates.AutoDateLocator(minticks=3,maxticks=10)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

def figure(fig, description):
    display(fig)
    plt.close(fig)
    say(description)

def chart_title(title):
    return f'{title}\n{period_label}'

def panel_grid(count, width=4.2, height=2.9):
    cols=min(3,count)
    return plt.subplots(math.ceil(count/cols),cols,figsize=(width*cols,height*math.ceil(count/cols)),
                        layout='constrained',squeeze=False)

def log_heatmap(ax, rows, years, title, unit):
    values=[[math.log10(1+rows[(year,m)]) if (year,m) in rows else math.nan for m in range(1,13)] for year in years]
    cmap=plt.colormaps['viridis'].with_extremes(bad='#e6e9ee')
    image=ax.imshow(values,aspect='auto',interpolation='nearest',cmap=cmap)
    ax.set(title=title, xlabel='Mois',ylabel='Année',xticks=range(12),xticklabels=range(1,13),
           yticks=range(len(years)),yticklabels=years)
    ax.figure.colorbar(image,ax=ax,label=f'log10(1 + {unit})')

def arrow_parquet():
    try:
        import pyarrow.parquet as pq
        return pq
    except ImportError:
        print('PyArrow indisponible dans ce noyau. Les graphiques fondés sur les Parquet nécessitent PyArrow.')
        return None

def parquet_years(path, column):
    # Les statistiques Parquet évitent la lecture des colonnes numériques. Lire l’horodatage
    # seulement si le groupe chevauche une année ou ne publie pas de bornes exploitables.
    pq=arrow_parquet()
    years=Counter()
    files=[]
    if pq is None:
        return years, files
    for item in sorted(path.rglob('*.parquet')):
        pf=pq.ParquetFile(item)
        if column not in pf.schema_arrow.names:
            continue
        size=item.stat().st_size
        count=pf.metadata.num_rows
        files.append((count,size))
        for i in range(pf.metadata.num_row_groups):
            group=pf.metadata.row_group(i)
            stats=next((group.column(j).statistics for j in range(group.num_columns)
                        if group.column(j).path_in_schema==column),None)
            if stats is not None and stats.has_min_max and hasattr(stats.min,'year') and stats.min.year==stats.max.year:
                years[stats.min.year]+=group.num_rows
            else:
                col=pf.read_row_group(i,columns=[column])[column]
                years.update(value.year for value in col.to_pylist() if value is not None)
    return years, files

## Paramètres du dataset

XAUUSD_DATASET_OPTIONS accepte du JSON pour choisir les variables, familles, horizons, dates, sources, seuils, normalisation et taille de fenêtre. Les options distinctes ont des sorties séparées.

In [4]:
if mode!='SNAPSHOT' and not all(completed(name) for name in ('gold','causality','tests_gold','catalogue_features')):
    raise RuntimeError('Gold complet est requis pour ce périmètre.')
dataset_options=json.loads(os.getenv('XAUUSD_DATASET_OPTIONS','{}'))
allowed_options={'main','higher','dataset_start','dataset_end','features','families','lookback','allowed_quality',
                 'dataset_sources','volume_required','scaler','split'}
if set(dataset_options)-allowed_options:
    raise ValueError(f'Options inconnues : {set(dataset_options)-allowed_options}')
if run_mode in {'FULL','PILOT'}:
    CFG['features']=['sma_5','sma_20','ema_5','ema_20','rsi_7','rsi_14','macd_5_13_4','atr_7',
                     'atr_14','bb_middle_20_2','tick_obv_delta_10','spread_bps']
CFG.update(dataset_options)
if CFG['main'] not in CFG['timeframes'] or not set(CFG['higher'])<=set(CFG['timeframes']):
    raise ValueError('Timeframes inconnus.')
if not isinstance(CFG['lookback'],int) or CFG['lookback']<1:
    raise ValueError('La longueur de fenêtre doit être un entier positif.')
if CFG['scaler'] not in {'standard','minmax','robust','none'}:
    raise ValueError('Normalisation inconnue.')
dataset_tag=digest(dataset_options)[:12] if dataset_options else 'default'
dataset_manifest='dataset' if not dataset_options else 'dataset_'+dataset_tag
if dataset_options:
    BUILT=BUILT.with_name(BUILT.name+'_'+dataset_tag)
print('Configuration Dataset Builder', {key:CFG.get(key) for key in sorted(allowed_options)})
print('Sortie', BUILT)

Configuration Dataset Builder {'allowed_quality': ['OK'], 'dataset_end': None, 'dataset_sources': ['historical_old', 'historical_recent', 'touhid', 'carlos', 'jforex', 'fixture'], 'dataset_start': None, 'families': None, 'features': ['sma_5', 'sma_20', 'ema_5', 'ema_20', 'rsi_7', 'rsi_14', 'macd_5_13_4', 'atr_7', 'atr_14', 'bb_middle_20_2', 'tick_obv_delta_10', 'spread_bps'], 'higher': ['M15', 'H1', 'H4', 'D1'], 'lookback': 1, 'main': 'M5', 'scaler': 'standard', 'split': None, 'volume_required': False}
Sortie /mnt/c/Users/omare/OneDrive/Desktop/Multi-Agents-Trading/data/pipeline_runs/full_notebooks/datasets/spark_xauusd


## Jointures causales

Chaque variable secondaire est disponible à compter de la clôture. Une absence ne provoque pas de remplissage vers le passé.

In [5]:
FREQ={'M1':60,'M5':300,'M15':900,'M30':1800,'H1':3600,'H4':14400,'D1':86400}

def with_higher(main,higher,tf,features,until):
    suffix='_'+tf
    good=higher.where(~F.col('is_partial'))
    w=Window.partitionBy('symbol').orderBy('available_at')
    close=F.col('available_at')
    if tf in FREQ:
        expiry=close+F.expr(f'INTERVAL {FREQ[tf]} SECONDS')
    elif tf=='W1':
        expiry=close+F.expr('INTERVAL 7 DAYS')
    else:
        expiry=F.add_months(close,1).cast('timestamp')
    limit=F.lit(until).cast('timestamp')+F.expr('INTERVAL 1 MICROSECOND')
    base=good.withColumn('next_time',F.least(F.coalesce(F.lead('available_at').over(w),limit),expiry))
    base=base.where('next_time > available_at').withColumn('join_date',F.explode(F.sequence(
        F.to_date('available_at'),F.to_date(F.col('next_time')-F.expr('INTERVAL 1 MICROSECOND')))))
    keep=[name for name in features if name in higher.columns]
    base=base.select('symbol','join_date',F.col('available_at').alias('lower'),F.col('next_time').alias('upper'),
        F.col('quality_flag').alias('quality'+suffix),*[F.col(x).alias(x+suffix) for x in keep])
    assigned=main.withColumn('join_date',F.to_date('available_at')).alias('m')
    other=base.alias('h')
    match=((F.col('m.symbol')==F.col('h.symbol')) & (F.col('m.join_date')==F.col('h.join_date')) &
           (F.col('m.available_at')>=F.col('h.lower')) & (F.col('m.available_at')<F.col('h.upper')))
    return (assigned.join(other,match,'left').select('m.*',
            F.col('h.lower').alias('available_at'+suffix), F.col('h.quality'+suffix),
            *[F.when(F.col('h.quality'+suffix).isin(*CFG['allowed_quality']),F.col('h.'+x+suffix)).alias(x+suffix) for x in keep])
            .drop('join_date'))

say('Une feature de fréquence supérieure conserve son horodatage de disponibilité et son statut de qualité.')

Une feature de fréquence supérieure conserve son horodatage de disponibilité et son statut de qualité.

## Fenêtres, séparation et normalisation

Une fenêtre est décrite par ses bornes et appartient à un seul split. Les paramètres standard, min-max ou robustes sont estimés exclusivement sur Train. Les prix Bid/Ask sont conservés.

In [6]:
def build_dataset():
    base=CFG['main']
    full=require(f'gold_{base}',GOLD/base).where(F.col('symbol')==CFG['instrument'])
    if CFG['dataset_start']:
        full=full.where(F.col('available_at')>=F.lit(CFG['dataset_start']).cast('timestamp'))
    if CFG['dataset_end']:
        full=full.where(F.col('available_at')<F.lit(CFG['dataset_end']).cast('timestamp'))
    catalog=read_json(reports/'feature_catalogue.json',[])
    if not catalog:
        raise ValueError('Catalogue Gold manquant.')
    available={r['feature'] for r in catalog if r['timeframe']==base}
    if CFG.get('families'):
        wanted=[r['feature'] for r in catalog if r['timeframe']==base and r['category'] in CFG['families']]
    elif CFG['features']=='all':
        wanted=sorted(available)
    else:
        wanted=list(CFG['features'])
    if not wanted or len(set(wanted))!=len(wanted):
        raise ValueError('Sélection vide ou dupliquée.')
    FEATURES=available
    for tf in CFG['higher']:
        if tf==base:
            continue
        secondary=require(f'gold_{tf}',GOLD/tf)
        requested=[key[:-len(tf)-1] for key in wanted if key.endswith('_'+tf)]
        if not requested:
            continue
        full=with_higher(full,secondary,tf,requested,str(end))
    if not set(wanted)<=set(full.columns):
        raise ValueError(f'Features absentes : {set(wanted)-set(full.columns)}')
    future=F.lit(False)
    for column in (name for name in full.columns if name.startswith('available_at_')):
        future=future | (F.col(column)>F.col('available_at'))
    assert not full.where(future).take(1)

    ok=(F.col('quality_flag').isin(*CFG['allowed_quality']) & ~F.col('is_partial') &
        F.forall('source_coverage',lambda item:item.isin(*CFG['dataset_sources'])))
    if CFG['volume_required']:
        ok=ok & (F.col('volume_coverage_ratio')==1)
    for name in wanted:
        x=F.col(name).cast('double')
        ok=ok & x.isNotNull() & ~F.isnan(x) & (F.abs(x)!=F.lit(float('inf')))
    look=CFG['lookback']
    permitted=full.withColumn('eligible',ok)
    if look>1:
        # Un segment est interrompu aux gaps par l’étape Gold. La fenêtre est calculée avant de filtrer.
        w=Window.partitionBy('symbol','segment').orderBy('available_at').rowsBetween(-look+1,0)
        permitted=permitted.withColumn('window_start',F.min('available_at').over(w)).where(
                  F.sum(F.col('eligible').cast('int')).over(w)==look)
    else:
        permitted=permitted.withColumn('window_start',F.col('available_at')).where('eligible')
    limite=permitted.agg(F.min('available_at'),F.max('available_at')).first()
    first,last=limite[0],limite[1]
    if first is None or last is None or first==last:
        say('**Dataset indisponible** · fenêtre trop courte ou warm-up insuffisant.')
        return None
    split=CFG['split']
    bounds=[first+(last-first)*p for p in (.70,.85)] if split is None else [datetime.fromisoformat(s) for s in split]
    assert len(bounds)==2 and bounds[0]<bounds[1]
    permitted=permitted.withColumn('split',F.when(F.col('available_at')<F.lit(bounds[0]),'train').when(
        F.col('available_at')<F.lit(bounds[1]),'validation').otherwise('test'))
    if look>1:
        permitted=permitted.where(
            ((F.col('split')=='train') & (F.col('window_start')<F.lit(bounds[0]))) |
            ((F.col('split')=='validation') & (F.col('window_start')>=F.lit(bounds[0]))) |
            ((F.col('split')=='test') & (F.col('window_start')>=F.lit(bounds[1]))))
    train=permitted.where(F.col('split')=='train')
    if not train.take(1):
        say('**Dataset indisponible** · Train est vide sur la fenêtre configurée.')
        return None
    scaler=CFG['scaler']
    assert scaler in {'standard','minmax','robust','none'}
    numeric={}
    if scaler!='none':
        if scaler=='robust':
            values=train.agg(*[F.percentile_approx(name,[.25,.5,.75],10000).alias(name)
                               for name in wanted]).first()
        else:
            fn=(F.avg,F.stddev_pop) if scaler=='standard' else (F.min,F.max)
            values=train.agg(*[operation(name).cast('double').alias(f'{i}_{j}')
                               for i,name in enumerate(wanted) for j,operation in enumerate(fn)]).first()
        for i,name in enumerate(wanted):
            if scaler=='robust':
                lower,location,upper=values[name]
                scale=upper-lower
            else:
                location,coefficient=values[f'{i}_0'],values[f'{i}_1']
                scale=(coefficient-location) if scaler=='minmax' else coefficient
            scale=scale or 1.0
            numeric[name]=[location,scale]
            permitted=permitted.withColumn('scaled_'+name,(F.col(name)-location)/scale)
    clocks=[name for name in full.columns if name.startswith('available_at_') or name.startswith('quality_') and name!='quality_flag']
    result=permitted.select('symbol','available_at','window_start','ask_close','bid_close','mid_close',
               'split','segment','quality_flag','source_coverage',*clocks,*wanted,
               *['scaled_'+name for name in numeric])
    result.explain('formatted')
    boundaries=F.when(F.col('split')=='validation',F.lit(bounds[0])).when(F.col('split')=='test',F.lit(bounds[1]))
    assert not result.where((F.col('window_start')>F.col('available_at')) |
                            (boundaries.isNotNull() & (F.col('window_start')<boundaries))).take(1)
    publish(result,BUILT,('split',))
    checked=spark.read.parquet(str(BUILT))
    counts={r['split']:r['count'] for r in checked.groupBy('split').count().limit(3).toLocalIterator()}
    split_details=[r.asDict() for r in checked.groupBy('split').agg(F.count('*').alias('observations'),
                  F.min('available_at').alias('first_utc'), F.max('available_at').alias('last_utc')).collect()]
    if run_mode in {'PILOT','FULL'} and set(counts)!={'train','validation','test'}:
        raise ValueError(f'Au moins un split est vide : {counts}')
    finish(dataset_manifest,rows=sum(counts.values()),splits=counts,cutoffs=[str(x) for x in bounds],
           columns=len(checked.columns),parquet_bytes=size_of(BUILT),split_details=split_details,
           features=wanted,scaler=scaler,scaler_train=numeric,lookback=look,
           window_reference=str((GOLD/base).relative_to(out)))
    return counts
if active('16') and not completed(dataset_manifest):
    if build_dataset() is None and run_mode!='TEST':
        raise ValueError('Dataset non publié : vérifier les fenêtres et les critères de qualité.')
else:
    status(dataset_manifest)

**dataset** · aucun résultat sauvegardé.

## Quality Gate, graphiques et tableau final

Le bilan affiche uniquement les mesures des manifestes compatibles. Les splits utilisent des dates UTC.

In [7]:
if completed(dataset_manifest):
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    record=manifest(dataset_manifest)
    details={r['split']:r for r in record['split_details']}
    labels=['train','validation','test']
    fig,ax=plt.subplots(figsize=(10,3.5),layout='constrained')
    for split in labels:
        if split in details:
            row=details[split]
            lo=datetime.fromisoformat(row['first_utc'])
            hi=datetime.fromisoformat(row['last_utc'])
            ax.plot([lo,hi],[split,split],marker='o',label=f"{split} · {row['observations']:,}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.set(title=f"Fenêtres admissibles · {CFG['main']} · {start} à {end} exclu",
           xlabel='Disponibilité UTC',ylabel='Split')
    ax.legend()
    display(fig)
    plt.close(fig)
    print('Chaque fenêtre est contenue dans son split. Le graphe indique les intervalles observés.')
    if spark:
        frame=require(dataset_manifest,BUILT)
        train=frame.where(F.col('split')=='train')
        keys=record['features'][:12]
        values=train.agg(*[F.corr(a,b).alias(f'{i}_{j}') for i,a in enumerate(keys)
                         for j,b in enumerate(keys)]).first().asDict()
        fig,ax=plt.subplots(figsize=(8,7),layout='constrained')
        image=ax.imshow([[values[f'{i}_{j}'] if values[f'{i}_{j}'] is not None else math.nan
                          for j in range(len(keys))] for i in range(len(keys))],vmin=-1,vmax=1,cmap='coolwarm')
        ax.set(title=f'Corrélation Pearson · Train · {CFG["main"]}',xticks=range(len(keys)),
               xticklabels=keys,yticks=range(len(keys)),yticklabels=keys)
        ax.tick_params(axis='x',rotation=65)
        fig.colorbar(image)
        display(fig)
        plt.close(fig)
        print('Aucune sélection automatique à partir de ce graphique. Les cellules vides peuvent représenter une constante.')
        # Corrélations de rang sur un ensemble borné et reproductible de Train.
        sample = train.orderBy(F.xxhash64('available_at'), F.col('available_at')).limit(5000)
        ranked = sample
        for i,key in enumerate(keys):
            w=Window.orderBy(F.col(key))
            tie=Window.partitionBy(F.col(key))
            ranked=ranked.withColumn(f'rank_{i}',F.rank().over(w)+(F.count('*').over(tie)-1)/2)
        spearman=ranked.agg(*[F.corr(f'rank_{i}',f'rank_{j}').alias(f'{i}_{j}')
                           for i in range(len(keys)) for j in range(len(keys))]).first().asDict()
        fig,ax=plt.subplots(figsize=(8,7),layout='constrained')
        image=ax.imshow([[spearman[f'{i}_{j}'] if spearman[f'{i}_{j}'] is not None else math.nan
                          for j in range(len(keys))] for i in range(len(keys))],vmin=-1,vmax=1,cmap='coolwarm')
        ax.set(title='Corrélation Spearman · jusqu’à 5 000 fenêtres Train',
               xticks=range(len(keys)),xticklabels=keys,yticks=range(len(keys)),yticklabels=keys)
        ax.tick_params(axis='x',rotation=65)
        fig.colorbar(image)
        display(fig)
        plt.close(fig)
        print('Les ex æquo reçoivent leur rang moyen. Les corrélations de rang sur un échantillon peuvent varier selon les périodes.')
        finish('tests_'+dataset_manifest,passed=['chronology','closed_higher','no_window_crosses_split',
                                              'finite_inputs','train_only_scaling'])
else:
    print('Aucun dataset compatible publié pour cette configuration.')

def cumulative_report():
    pairings=[
        ('Bronze','bronze',None,'rows',out/'bronze/profile'),
        ('Cleaning','silver','raw','accepted',NORM),
        ('Normalization','silver','raw','accepted',NORM),
        ('Integration','silver','accepted','silver',TICKS),
        ('Silver Ticks','validation',None,'ticks',TICKS),
        ('OHLCV','bars',None,'counts',BARS),
        ('Gold','gold',None,'features',GOLD),
        ('Dataset Builder',dataset_manifest,None,'rows',BUILT)]
    rows=[]
    silver=manifest('silver')
    ticks=silver.get('silver')
    bar_counts=manifest('bars').get('counts',{})
    def measured_partitions(target):
        paths=list(target.rglob('*.parquet'))
        return len(paths),len({str(path.parent.relative_to(target)) for path in paths})
    phase_entries={'Bronze':manifest('bronze').get('rows'),'Silver Ticks':ticks,
                   'OHLCV':ticks,'Gold':sum(bar_counts.values()) if bar_counts else None,
                   'Dataset Builder':manifest(f"gold_{CFG['main']}").get('rows')}
    for label,key,read_key,write_key,target in pairings:
        stats=manifest(key)
        if stats.get('run_key')!=run_key:
            rows.append(dict(stage=label,status='non exécuté'))
            continue
        entries=stats.get(read_key) if read_key else phase_entries.get(label)
        written=stats.get(write_key)
        if key=='gold':
            written=sum(manifest('gold_'+tf).get('rows',0) for tf in CFG['timeframes'])
        if key=='bars':
            written=sum(bar_counts.values())
        files,partitions=measured_partitions(target)
        columns=stats.get('columns')
        if spark and files and label not in {'Bronze','OHLCV','Gold'} and not columns:
            files_hint=list(target.rglob('*.parquet'))
            columns=len(spark.read.parquet(str(files_hint[0])).columns)
        summary=dict(stage=label,status='mesuré',rows_in=entries,rows_out=written,
                     columns=columns,files=files,physical_partitions=partitions,
                     bytes=size_of(target),seconds=stats.get('elapsed_run_seconds'))
        rows.append(summary)
    if spark:
        put_json(reports/f'rapport_final_{dataset_tag}.json',dict(mode=run_mode,start=str(start),end=str(end),
            stages=rows,by_year=read_json(reports/'silver_annuel.json',[]),by_timeframe=manifest('bars').get('counts')))
    print('Durées cumulées dans le notebook qui a produit chaque manifeste.')
    keys=('stage','rows_in','rows_out','columns','physical_partitions','bytes','seconds','status')
    headings=('Étape','Lignes entrée','Lignes sortie','Colonnes','Partitions physiques','Octets','Temps cumulé (s)','Statut')
    table=['| '+' | '.join(headings)+' |','| '+' | '.join(['---']*len(keys))+' |']
    table.extend('| '+' | '.join(str(row.get(k)) if row.get(k) is not None else 'indisponible' for k in keys)+' |' for row in rows)
    say('\n'.join(table))
cumulative_report()
stage_report()

Aucun dataset compatible publié pour cette configuration.
Durées cumulées dans le notebook qui a produit chaque manifeste.


| Étape | Lignes entrée | Lignes sortie | Colonnes | Partitions physiques | Octets | Temps cumulé (s) | Statut |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Bronze | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Cleaning | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Normalization | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Integration | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Silver Ticks | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| OHLCV | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Gold | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |
| Dataset Builder | indisponible | indisponible | indisponible | indisponible | indisponible | indisponible | non exécuté |

Spark UI : métriques de tâches indisponibles.
Métriques Spark UI {}
dataset non exécuté ou incompatible


## Partitions d’entraînement, validation et test

Ajouter au calendrier des splits des vues par année, par volume et par taille des fichiers Parquet.

In [8]:
def dataset_layout_charts():
    if not completed(dataset_manifest):
        print('Les graphiques Train, Validation et Test attendent un dataset FULL publié.')
        return
    state=manifest(dataset_manifest)
    labels=[name for name in ('train','validation','test') if name in state['splits']]
    fig,axs=plt.subplots(1,3,figsize=(13,3.7),layout='constrained')
    all_years={}
    physical={}
    for label in labels:
        path=BUILT/f'split={label}'
        all_years[label],physical[label]=parquet_years(path,'available_at')
    counts=[state['splits'][label] for label in labels]
    sizes=[sum(p.stat().st_size for p in (BUILT/f'split={name}').rglob('*.parquet'))/mib for name in labels]
    for ax,values,title,unit in [(axs[0],counts,'Observations admissibles','Lignes'),
                                 (axs[1],sizes,'Taille physique','Mio de Parquet'),
                                 (axs[2],[100*n/sum(counts) for n in counts],'Part du dataset','% des lignes')]:
        ax.bar(labels,values,color=['#157477','#17477d','#b96026'][:len(labels)])
        ax.set(title=title,xlabel='Split chronologique',ylabel=unit)
    fig.suptitle(chart_title(f'Dataset · {CFG["main"]} · {len(state["features"])} variables'))
    figure(fig,'Les décomptes concernent les fenêtres admissibles publiées. Les jeux suivent l’ordre chronologique.')
    years=sorted({y for counts in all_years.values() for y in counts})
    if years:
        fig,ax=plt.subplots(figsize=(11,4),layout='constrained')
        bottom=[0]*len(years)
        for name in labels:
            values=[all_years[name][y] for y in years]
            ax.bar(years,values,bottom=bottom,label=name)
            bottom=[a+b for a,b in zip(bottom,values)]
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set(title=chart_title('Observations finales par année'),xlabel='Année UTC',ylabel='Lignes')
        ax.legend()
        figure(fig,'Si une coupure se situe au milieu de l’année, plusieurs couleurs apparaissent dans la même barre.')
    if any(physical.values()):
        fig,ax=plt.subplots(figsize=(9,3.8),layout='constrained')
        for name in labels:
            if physical[name]:
                ax.scatter([n for n,_ in physical[name]],[s/mib for _,s in physical[name]],alpha=.75,label=name)
        ax.set(title=chart_title('Répartition des fichiers du dataset'),xlabel='Lignes par fichier',ylabel='Mio par fichier')
        ax.legend()
        figure(fig,'Chaque point est un fichier Parquet publié sous une partition split=… .')
dataset_layout_charts()

Les graphiques Train, Validation et Test attendent un dataset FULL publié.
